In [29]:
import json
import pandas as pd

with open("../data/processed/questions.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

df.head()

,id,domain,difficulty,question,reference_answer
0,1,DATA ANALYSIS,Easy,What is R2? What are some other metrics that c...,goodness of fit measure. variance explained by...
1,2,DATA ANALYSIS,Medium,What is the curse of dimensionality?,"High dimensionality makes clustering hard, bec..."
2,3,DATA ANALYSIS,Easy,Is more data always better?,"- Statistically,\nIt depends on the quality of..."
3,4,DATA ANALYSIS,Easy,What are advantages of plotting your data befo...,Data sets have errors. You won't find them al...
4,5,DATA ANALYSIS,Medium,How can you make sure that you don’t analyze s...,Proper exploratory data analysis.\nIn every da...


In [30]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn Names:")
print(df.columns.tolist())

Rows: 998
Columns: 5

Column Names:
['id', 'domain', 'difficulty', 'question', 'reference_answer']


In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 998 entries, 0 to 997
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   id                998 non-null    int64
 1   domain            998 non-null    str  
 2   difficulty        998 non-null    str  
 3   question          998 non-null    str  
 4   reference_answer  998 non-null    str  
dtypes: int64(1), str(4)
memory usage: 39.1 KB


In [32]:
print(df.isnull().sum())

id                  0
domain              0
difficulty          0
question            0
reference_answer    0
dtype: int64


In [33]:
blank_questions = (
    df["question"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Blank Questions:", blank_questions)

Blank Questions: 0


In [34]:
blank_answers = (
    df["reference_answer"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Blank Answers:", blank_answers)

Blank Answers: 4


In [35]:
blank_difficulty = (
    df["difficulty"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Blank Difficulty:", blank_difficulty)

Blank Difficulty: 2


In [36]:
print("Total Questions:", len(df))

print(
    "Unique Questions:",
    df["question"].nunique()
)

Total Questions: 998
Unique Questions: 526


In [37]:
duplicate_count = (
    len(df)
    - df["question"].nunique()
)

print(
    "Duplicate Questions:",
    duplicate_count
)

Duplicate Questions: 472


In [38]:
duplicates = df[
    df.duplicated(
        subset=["question"],
        keep=False
    )
]

duplicates[
    ["id","question"]
].sort_values("question")

,id,question
747,748,Explain inheritance in Python with an example.
781,782,Explain inheritance in Python with an example.
780,781,Explain inheritance in Python with an example.
779,780,Explain inheritance in Python with an example.
778,779,Explain inheritance in Python with an example.
...,...,...
643,644,What is the time complexity of binary search?
705,706,What is the time complexity of binary search?
709,710,What is the time complexity of binary search?
681,682,What is the time complexity of binary search?


In [39]:
print(
    df["domain"]
    .value_counts()
)

domain
Python                  500
ML                      257
DL                      110
NLP                      41
PREDICTIVE MODELLING     34
SQL                      30
DATA ANALYSIS            26
Name: count, dtype: int64


In [40]:
print(
    df["difficulty"]
    .value_counts()
)

difficulty
Medium    465
Easy      321
Hard      210
            2
Name: count, dtype: int64


In [41]:
duplicates = df[
    df.duplicated(
        subset=["question"],
        keep=False
    )
]

print("Duplicate Records:", len(duplicates))

Duplicate Records: 500


In [42]:
df_clean = df.drop_duplicates(
    subset=["question"],
    keep="first"
)

In [45]:
domain_dist = (
    df_clean["domain"]
    .value_counts()
    .reset_index()
)

domain_dist.columns = [
    "Domain",
    "Count"
]

domain_dist

,Domain,Count
0,ML,257
1,DL,110
2,NLP,41
3,PREDICTIVE MODELLING,34
4,SQL,30
5,Python,28
6,DATA ANALYSIS,26


In [59]:
difficulty_dist = (
    df_clean["difficulty"]
    .value_counts()
    .reset_index()
)

difficulty_dist.columns = [
    "Difficulty",
    "Count"
]

difficulty_dist

,Difficulty,Count
0,Medium,235
1,Easy,171
2,Hard,118
3,,2


In [48]:
domain_difficulty = pd.crosstab(
    df_clean["domain"],
    df_clean["difficulty"]
)

domain_difficulty

difficulty,,Easy,Hard,Medium
domain,,,,
DATA ANALYSIS,1,4,8,13
DL,0,55,14,41
ML,1,58,61,137
NLP,0,27,5,9
PREDICTIVE MODELLING,0,7,12,15
Python,0,10,8,10
SQL,0,10,10,10


In [49]:
domain_summary = pd.DataFrame({
    "Total Questions":
        df_clean.groupby("domain")["question"].count(),

    "Unique Questions":
        df_clean.groupby("domain")["question"].nunique()
})

domain_summary

,Total Questions,Unique Questions
domain,,
DATA ANALYSIS,26,26
DL,110,110
ML,257,257
NLP,41,41
PREDICTIVE MODELLING,34,34
Python,28,28
SQL,30,30


In [50]:
df_clean["answer_length"] = (
    df_clean["reference_answer"]
    .fillna("")
    .astype(str)
    .str.split()
    .str.len()
)

In [51]:
print(
    "Average Answer Length:",
    round(
        df_clean["answer_length"].mean(),
        2
    )
)

Average Answer Length: 46.33


In [52]:
df_clean.sort_values(
    "answer_length"
)[
    [
        "id",
        "question",
        "answer_length"
    ]
].head(10)

,id,question,answer_length
20,21,You have 100 mathletes and 100 math problems. ...,0
402,403,Explain Lemmatization.,0
414,415,What is Pragmatic Ambiguity?,0
404,405,Explain Named Entity Recognition.,0
421,422,Which NLP model gives the best accuracy?,1
9,10,Let’s say you’re given an unfeasible amount of...,2
476,477,What is a NULL value?,5
486,487,What is a self join?,5
485,486,Explain ACID properties.,5
495,496,What is sharding?,5


In [53]:
df_clean.sort_values(
    "answer_length",
    ascending=False
)[
    [
        "id",
        "question",
        "answer_length"
    ]
].head(10)

,id,question,answer_length
70,71,"What do you understand by Perceptron? Also, ex...",1326
66,67,What are the issues faced while training in Re...,276
64,65,What do you understand by a convolutional neur...,197
121,122,What are some advantages in using a CNN (convo...,196
13,14,What is the main idea behind ensemble learning...,190
253,254,How would you evaluate a logistic regression m...,185
153,154,What is Confusion Metrics,185
123,124,Suppose you have a NN with 3 layers and ReLU a...,184
88,89,How can hyperparameters be trained in neural n...,148
407,408,What is Latent Semantic Indexing (LSI)?,147


In [54]:
df_clean.groupby(
    "domain"
)["answer_length"].mean().round(2)

domain
DATA ANALYSIS           49.08
DL                      72.65
ML                      42.42
NLP                     65.29
PREDICTIVE MODELLING    15.47
Python                  27.39
SQL                      7.70
Name: answer_length, dtype: float64

In [55]:
df_clean.groupby(
    "difficulty"
)["answer_length"].mean().round(2)

difficulty
           7.50
Easy      49.43
Hard      49.96
Medium    42.59
Name: answer_length, dtype: float64